# Tutorial 03 — Soft mobility tensors and simulation of a trajectory

A *soft body* in SoftMobility differs from a rigid one by carrying
**internal degrees of freedom** (DOFs) that the solver integrates in
time alongside the body position and orientation. Each DOF is paired
with an elastic restoring force/torque (and optional input couplings),
and the framework automatically derives the soft mobility tensors

$$\mathbf{M},\ \mathbf{M}_K,\ \mathbf{M}_H,\ \mathbf{C}_E,\ \mathbf{P}$$

that map external loads, DOF deformation, and ambient flow strain to
the time derivatives of the body coordinates and DOFs.

This tutorial:

1. builds a one-DOF deformable dumbbell with a torsion spring,
2. inspects the soft mobility tensors at the default configuration,
3. simulates the body falling under gravity, watching the DOF relax
   back to its equilibrium while the body translates.


In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import softmobility as sm

np.set_printoptions(precision=3, suppress=True)


## Step 1 — A one-DOF deformable dumbbell

Two unit-density spheres on either side of the body origin. The bending
DOF `theta0` rotates each sphere antisymmetrically around `ê_y`, and a
torsion spring of stiffness `k` provides a linear restoring torque.


In [2]:
yaml_body = """
dof_names:    [theta]
design_names: [k, length]
input_names:  [gravity]

defaults:
  theta0: 0.5    # initial tilt (radians)
  k:      5.0
  length: 1.0

spheres:
  - radius: 0.5
    position:    [-length, 0, 0]
    orientation: [0, theta0, 0]
    force:       [0.523599 * gravity0, 0.523599 * gravity1, 0.523599 * gravity2]
    torque:      [0, -k * theta0, 0]
  - radius: 0.5
    position:    [length, 0, 0]
    orientation: [0, -theta0, 0]
    force:       [0.523599 * gravity0, 0.523599 * gravity1, 0.523599 * gravity2]
    torque:      [0, k * theta0, 0]
"""

body = sm.SoftBody(yaml_body, verbose=False)
print(repr(body))


SPHERE ASSEMBLY
  2 spheres
  1 degrees of freedom
  2 design parameters
  3 input parameters

Default values
  degrees of freedom dof: ['theta0'] = [0.5]
  design parameters param: ['k', 'length'] = [5. 1.]
  input parameters param: ['gravity0', 'gravity1', 'gravity2']

SPHERE 0
  radius: 0.5
  position: [-1.  0.  0.]
  orientation: [0.  0.5 0. ]
  C_H:
[[0.524 0.    0.   ]
 [0.    0.524 0.   ]
 [0.    0.    0.524]
 [0.    0.    0.   ]
 [0.    0.    0.   ]
 [0.    0.    0.   ]]
  C_K:
[[ 0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [-5.]
 [ 0.]]

SPHERE 1
  radius: 0.5
  position: [1. 0. 0.]
  orientation: [ 0.  -0.5  0. ]
  C_H:
[[0.524 0.    0.   ]
 [0.    0.524 0.   ]
 [0.    0.    0.524]
 [0.    0.    0.   ]
 [0.    0.    0.   ]
 [0.    0.    0.   ]]
  C_K:
[[0.]
 [0.]
 [0.]
 [0.]
 [5.]
 [0.]]



## Step 2 — The soft mobility tensors

`compute_tensors()` returns a named tuple with five blocks. Their shapes
follow the body dimensions: `Nspheres`, `Ndof`, `Ninput`.


In [3]:
tensors = body.compute_tensors()
print("shapes:")
print("  M       (body-frame mobility for sums of forces+torques)  ::", tensors.M.shape)
print("  M_K     (DOF -> body translational+rotational+DOF rates)   ::", tensors.M_K.shape)
print("  M_H     (inputs -> body translational+rotational+DOF rates)::", tensors.M_H.shape)
print("  C_E     (ambient strain coupling)                          ::", tensors.C_E.shape)
print("  P       (projector onto reduced velocity space)            ::", tensors.P.shape)
print()
print("M_H (gravity sensitivity):\n", tensors.M_H)
print()
print("M_K (DOF sensitivity):\n", tensors.M_K)


shapes:
  M       (body-frame mobility for sums of forces+torques)  :: (7, 12)
  M_K     (DOF -> body translational+rotational+DOF rates)   :: (7, 1)
  M_H     (inputs -> body translational+rotational+DOF rates):: (7, 3)
  C_E     (ambient strain coupling)                          :: (7, 5)
  P       (projector onto reduced velocity space)            :: (7, 12)

M_H (gravity sensitivity):
 [[ 0.076  0.     0.   ]
 [ 0.     0.066  0.   ]
 [ 0.     0.     0.066]
 [ 0.     0.     0.   ]
 [ 0.     0.     0.   ]
 [ 0.    -0.     0.   ]
 [ 0.     0.    -0.005]]

M_K (DOF sensitivity):
 [[ 0.   ]
 [ 0.   ]
 [ 0.05 ]
 [ 0.   ]
 [ 0.   ]
 [ 0.   ]
 [-1.604]]


## Step 3 — Simulate a trajectory under gravity

Gravity along `-ẑ` (pulls the body down). The DOF starts at its default
value `0.5` rad of bend; the spring drives it back toward `0`.


In [4]:
rollout = sm.FlowBodyRollout(
    soft_body=body,
    flow=sm.no_flow(),
    input_map={"gravity": sm.gravity_field(g=9.81)},
)

dt = 0.01
n_steps = 600
positions, orientations, dofs = rollout.rollout(dt=dt, n_steps=n_steps)
positions.block_until_ready()
print("positions shape:", positions.shape)
print("dofs shape    :", dofs.shape)
print("final dof     :", dofs[-1])
print("final z       :", float(positions[-1, 2]))


positions shape: (600, 3)
dofs shape    : (600, 1)
final dof     : [0.032]
final z       : -3.8846540451049805


## Step 4 — Plot the trajectory and the DOF

The body falls along `−ẑ` while the spring relaxes the bending DOF back
toward zero. The two motions are **coupled**: the soft mobility tensor
`M_H` carries the gravity force into a small DOF rate, and `M_K` carries
the elastic restoring torque (proportional to `theta0`) into both DOF
relaxation and a transient body translation.


In [5]:
t = np.arange(n_steps) * dt

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Body z-position", "Bending DOF θ₀"),
    horizontal_spacing=0.15,
)
fig.add_trace(go.Scatter(x=t, y=np.asarray(positions[:, 2]),
                         mode="lines", line=dict(color="#1f77b4", width=2),
                         showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=np.asarray(dofs[:, 0]),
                         mode="lines", line=dict(color="#d62728", width=2),
                         showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="time", row=1, col=1)
fig.update_xaxes(title_text="time", row=1, col=2)
fig.update_yaxes(title_text="z(t)", row=1, col=1)
fig.update_yaxes(title_text="θ₀(t)  [rad]", row=1, col=2)
fig.update_layout(width=900, height=400, plot_bgcolor="white")
fig.show()


## Notes

* `compute_tensors()` is a **single forward evaluation** at the supplied
  `dofs`, `design`, and `time`. The full nonlinear hydrodynamic problem
  is reduced once at every time step inside the rollout.
* The tensors are JAX-compatible — `jax.jit`, `jax.grad`, and `jax.vmap`
  all work, so derivatives with respect to design parameters or initial
  conditions are available for free. Tutorial **04 — optimization**
  uses this for gradient-based design.
* For multi-DOF bodies (swimmers, fibers), the same workflow applies —
  the tensor shapes simply grow with `Ndof`. See tutorial **22 —
  three-sphere swimmer** for a 4-DOF active body.
